In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

from armored.models import *
from armored.preprocessing import *

import itertools
from tqdm import tqdm

import shap

/home/jcthompson5@ad.wisc.edu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")
df_exp4a = pd.read_csv("data/exp4/exp4_metabolites_best_reps.csv")
df_exp4b = pd.read_csv("data/exp4/exp4_metabolites_new_best.csv")
df_exp4c = pd.read_csv("data/exp4/exp4_metabolites_new_worst.csv")
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3, df_exp4a, df_exp4b, df_exp4c))

# define variable names
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

# average replicates
df_fmt_mean = []
for exp_name, df_exp in df.groupby("Experiments"):
    df_groups = df_exp.groupby("Time")
    df_avg = df_groups[system_variables].mean().reset_index()
    df_avg.insert(0, "Experiments", [exp_name]*df_avg.shape[0])
    df_fmt_mean.append(df_avg)
df = pd.concat(df_fmt_mean)

In [3]:
# log that ignores zeros
def zlog(x):
    x[x <= 0] = 1
    return np.log(x)

# shannon diversity
def shannon(y):
    y_rel = y / np.sum(y)
    return np.nansum(-zlog(y_rel)*y_rel)

# define objective 
def objective(y):
    # y is measured exp data [n_time, n_species + n_metabolites]
    
    # endpoint shannon diversity
    diversity = shannon(y[-1, :len(species)])
    
    # variance in species abundances in last two passages
    if np.any(np.isnan(y[-2:, :len(species)])):
        instability = np.nan
    else:
        species_stdv = np.std(y[-2:, :len(species)], 0)
        instability  = np.where(species_stdv>0, species_stdv, 0).mean() 
    
    # endpoint butyrate production 
    butyrate =  y[-1, -2]   
    
    return diversity, instability, butyrate

In [ ]:
# determine names of experimental conditions 
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(df)
df_scaled = scaler.transform(df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
train_data = format_data(df, species, metabolites, controls, observed=observed)
train_data_scaled = format_data(df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls), 
             n_hidden=32)

# fit model
brnn.fit(train_data_scaled, 
         alpha_0=0, alpha_1=1.,
         evd_tol=1e-3)

Total measurements: 27244, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1353.324, Residuals: -0.00492
Loss: 1268.678, Residuals: -0.00160
Loss: 1238.723, Residuals: 0.00099
Loss: 1209.082, Residuals: 0.00157
Loss: 1096.400, Residuals: 0.00427
Loss: 1069.116, Residuals: -0.00230
Loss: 1020.511, Residuals: -0.00057
Loss: 936.307, Residuals: -0.00050
Loss: 901.252, Residuals: -0.00078
Loss: 837.994, Residuals: 0.00022
Loss: 775.194, Residuals: -0.00014
Loss: 756.559, Residuals: -0.00078
Loss: 753.812, Residuals: -0.00079
Loss: 731.881, Residuals: -0.00079
Loss: 690.453, Residuals: -0.00138
Loss: 627.972, Residuals: -0.00147
Loss: 625.930, Residuals: -0.00145
Loss: 622.027, Residuals: -0.00123
Loss: 615.249, Residuals: -0.00089
Loss: 602.518, Residuals: -0.00093
Loss: 580.242, Residuals: -0.00105
Loss: 572.157, Residuals: -0.00192
Loss: 558.007, Residuals: -0.00153
Loss: 537.026, Residuals: -0.00146
Loss: 525.342, Residuals: -0.00136
Loss: 524.758, Residuals: -0.00132

In [ ]:
# concatenate data points
Xs = []
all_exp_names = []

for (T, X, U, Y, exp_names) in train_data_scaled:
    
    all_exp_names.append(exp_names)
    for xi, ui in zip(X, U):
        
        # append design condition
        Xs.append(np.append(xi, ui[0]))
        
# stack 
X = np.stack(Xs)  
all_exp_names = np.concatenate(all_exp_names)

# init SHAP explainer with baseline
baseline = np.zeros([1, X.shape[-1]])
# baseline is zero for species, metabolites and fibers but pH is always normalized to 1
baseline[0,list(system_variables).index('pH')] = 1.

In [ ]:
# set each species as receiver
for i, receiver in enumerate(species):

    # create wrapper for brnn to match SHAP model 
    def model(X):

        # X is matrix of [n_samples, n_inputs] 
        # Decompose X into Species/Metabolites and Fibers
        Xsm = X[:, :len(observed)]
        Xf = X[:, len(observed):]

        # matrix of predictions over time
        Xf = np.stack([np.stack(4*[Xfi]) for Xfi in Xf])
        Y = brnn.forward_batch(brnn.params, Xsm, Xf)

        # return endpoint species prediction
        return Y[:, -1, i]
    
    # compute the SHAP values for the model
    feature_names = np.array([v.split("abs")[0] for v in system_variables])
    explainer = shap.Explainer(model, baseline, feature_names=feature_names)
    shap_values = explainer(X)

    # name of shap explanations
    shap_features = []
    for j, affecter in enumerate(system_variables):
        shap_features.append(receiver + "<--" + affecter)

    # init df to save shap values
    df_sensitivity = pd.DataFrame()
    df_sensitivity["Experiments"] = all_exp_names

    # add shap values
    for j, shap_feature in enumerate(shap_features):
        df_sensitivity[shap_feature] = shap_values.values[:, j]

    # add design conditions
    for j, sys_var in enumerate(system_variables):
        if sys_var not in metabolites:
            df_sensitivity[sys_var] = X[:, j]
        
    # save df 
    df_sensitivity.to_csv(f"insights/community_shap/{receiver}_shap.csv", index=False)